# Session 13 — 1/3: setup, correctness, baseline

Trains the ~20M model for 50M tokens **without** reversibility, at the largest
batch this GPU can hold. That batch is then reused unchanged by notebook 2 so
the arms are comparable.

In [ ]:
!nvidia-smi
!git clone https://github.com/rjvim/era-v5-session13-reversibility repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, 'src')

## Correctness first

The custom activation-free backward is checked against ordinary autograd in float64 before any timing number is trusted.

In [ ]:
!pytest tests/test_reversibility.py -q

## Data — 50M training tokens (+1M val), GPT-2 encoding

Written once as a uint16 memmap so every arm reads identical tokens in identical order.

In [ ]:
!python src/data.py --out_dir data --tokens 50000000
!ls -la data/

## Largest batch the baseline can hold

Doubling + binary search over a *full* train step (fwd, bwd, optimiser), not a forward pass.

In [ ]:
!python src/train.py --mode baseline --find_max_batch --seq_len 512

## Run 1 — baseline @ fixed batch

In [ ]:
import json
BATCH_FIX = json.load(open('results/maxbatch_baseline.json'))['max_batch']
print('fixed batch =', BATCH_FIX)
!python src/train.py --mode baseline --batch_size {BATCH_FIX}     --run_name baseline_fixed --seq_len 512 --total_tokens 50000000 --resume


In [ ]:
d = json.load(open('results/baseline_fixed.json'))
print(f"loss {d['final_train_loss']:.4f} | val {d['final_val_loss']:.4f} | "
      f"{d['tok_per_s']:,.0f} tok/s | peak {d['peak_mem_gb']:.2f} GB")

Carry `BATCH_FIX` into notebook 2 unchanged.